# DSTFGCN on PEMS-BAY (Kaggle) - comparison baseline for HGP

Runs the authors' DSTFGCN code (SYLan2019/DSTFGCN, Neural Networks 2026) **unchanged** (`train.py`, `engine.py`, `model/`, `util.py`) on PEMS-BAY with the same chronological 70/10/20 split, the same road graph (DCRNN `adj_mx_bay.pkl`) and the same common training settings as HGP, and reports test MAE / RMSE / MAPE at 15, 30 and 60 minutes in HGP's format.

Added files: `prepare_pemsbay.py` (data, using the authors' own windowing function), `run_hgp_protocol.py` (swaps the optimiser and schedule, then runs `train.py`), `evaluate_hgp_metrics.py` (HGP-style test metrics from the best checkpoint).

**Before running**: GPU on (T4 / P100), Internet On, add a dataset holding `pems-bay.h5` and `adj_mx_bay.pkl`.

In [ ]:
SEED = 42
MAX_EPOCH = 40            # HGP protocol; the authors train 200
REPO_URL = "https://github.com/SYLan2019/DSTFGCN.git"   # the authors' repository (no licence file: used here only to run the experiment)


## 1. Environment, code, data

In [ ]:
import glob, json, os, shutil, subprocess, sys
import torch
print(sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "turn the GPU accelerator on"
WORK, CODE = "/kaggle/working", "/tmp/dstfgcn"
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
os.chdir(CODE)
# the authors' repository may carry its own data folders; the run uses data/PEMS-BAY written below
files = [f for f in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(f)]
TRAFFIC = [f for f in files if f.lower().endswith(".h5") and "pems" in os.path.basename(f).lower() and "meta" not in os.path.basename(f).lower()][0]
ADJ = [f for f in files if f.lower().endswith(".pkl") and "adj" in os.path.basename(f).lower()][0]
print(TRAFFIC, ADJ)


In [ ]:
%%writefile prepare_pemsbay.py
"""PEMS-BAY -> the input files DSTFGCN's train.py reads. Data preparation only; no model code is touched.

    python prepare_pemsbay.py --traffic pems-bay.h5 --adjacency adj_mx_bay.pkl          (run inside the DSTFGCN repository)

Writes data/PEMS-BAY/{train,val,test}.npz and data/graph/adj_PEMS-BAY.pkl.

It follows the authors' pre_process_data.py (generate_seq / generate_from_data): the series is cut chronologically first, 12-step input
windows and the following 12-step targets are cut inside each part, the inputs are standardised with the mean/std of the
training inputs, and the targets stay in the original units. Differences from the authors' script, all for fairness with HGP:
  * the cut points are HGP's: train = first int(0.7 T) rows, validation = next int(0.1 T), test = the rest (the authors use 60/20/20);
  * missing readings (0 in PEMS-BAY) stay 0 in the inputs - the model's loss and metrics treat 0 as missing.
The road graph is DCRNN's adj_mx_bay.pkl, the one HGP uses, stored as [sensor_ids, id_to_index, adj] because train.py reads adj[2].
"""
import argparse
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from pre_process_data import generate_seq          # the authors' windowing function

ap = argparse.ArgumentParser()
ap.add_argument("--traffic", required=True)
ap.add_argument("--adjacency", required=True)
ap.add_argument("--train_rate", type=float, default=0.7)
ap.add_argument("--val_rate", type=float, default=0.1)
ap.add_argument("--sensors", type=int, default=None, help="keep the first K sensors (smoke test only)")
ap.add_argument("--steps", type=int, default=None, help="keep the first K time steps (smoke test only)")
a = ap.parse_args()

table = pd.read_hdf(a.traffic)
with open(a.adjacency, "rb") as f:
    content = pickle.load(f, encoding="latin1")
ids, id_to_index, adj = content[0], content[1], np.asarray(content[2], dtype=np.float32)
if a.sensors:
    table, adj, ids = table.iloc[:, :a.sensors], adj[:a.sensors, :a.sensors], ids[:a.sensors]
if a.steps:
    table = table.iloc[:a.steps]
data = np.nan_to_num(table.values.astype(np.float32), nan=0.0)[:, :, None]       # [T, N, 1]
T, N, _ = data.shape

train_end = int(T * a.train_rate)
val_end = train_end + int(T * a.val_rate)
parts = {"train": data[:train_end], "val": data[train_end:val_end], "test": data[val_end:]}

out = Path("data/PEMS-BAY")
out.mkdir(parents=True, exist_ok=True)
mean = std = None
for name, part in parts.items():
    x, y = generate_seq(part, 12, 12)                                    # x, y: [windows, 12, N, 1]
    if mean is None:                                                     # as the authors: statistics of the training inputs
        mean, std = x.mean(), x.std()
    np.savez_compressed(out / f"{name}.npz", x=((x - mean) / std).astype(np.float32), y=y.astype(np.float32))
    print(f"{name}: rows {len(part)}, windows {len(x)}, x {x.shape}")

Path("data/graph").mkdir(parents=True, exist_ok=True)
with open("data/graph/adj_PEMS-BAY.pkl", "wb") as f:
    pickle.dump([ids, id_to_index, adj], f)
print(f"{T} steps x {N} sensors, {(data == 0).mean():.4%} missing, train mean {mean:.3f} std {std:.3f}, {(adj != 0).sum()} road links")


In [ ]:
%%writefile run_hgp_protocol.py
"""Runs the authors' train.py unchanged, with HGP's optimiser and learning-rate schedule.

engine.py hard-codes Adam and a MultiStepLR that train.py steps once per epoch. HGP trains with AdamW, one warm-up epoch and cosine
decay to zero, stepped once per batch. This wrapper swaps exactly those two objects before train.py runs; the model, the loss
(masked Huber), gradient clipping at 5, the data loader and the model selection (best validation loss, early stopping) are the authors' own.

    python run_hgp_protocol.py --config_path config/PEMS-BAY_hgp.json --device cuda:0
"""
import math
import runpy
import sys

import torch

sys.path.insert(0, ".")
import util
import engine

steps = {}
_load = util.load_dataset


def load_dataset(*args, **kwargs):
    data = _load(*args, **kwargs)
    steps["per_epoch"] = data["train_loader"].num_batch
    return data


util.load_dataset = load_dataset


class PerBatchSchedule:
    """HGP's schedule (hgp/train.py): linear warm-up for `warmup_epochs`, then cosine decay to 0."""

    def __init__(self, optimizer, max_epoch, warmup_epochs=1):
        total = max_epoch * steps["per_epoch"]
        warm = max(1, warmup_epochs * steps["per_epoch"])

        def factor(step):
            if step < warm:
                return (step + 1) / warm
            return 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm))))

        self.inner = torch.optim.lr_scheduler.LambdaLR(optimizer, factor)

    def step(self):          # train.py calls this once per epoch: ignored, the schedule advances per batch
        pass


_init, _train = engine.trainer.__init__, engine.trainer.train


def init(self, config, scaler, adjacency_matrix, device):
    torch.optim.Adam = lambda params, lr, weight_decay: torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    torch.optim.lr_scheduler.MultiStepLR = lambda optimizer, **kw: PerBatchSchedule(optimizer, config["max_epoch"])
    _init(self, config, scaler, adjacency_matrix, device)


def train(self, input, real_val, time_i):
    result = _train(self, input, real_val, time_i)
    self.scheduler.inner.step()
    return result


engine.trainer.__init__, engine.trainer.train = init, train
runpy.run_path("train.py", run_name="__main__")


In [ ]:
%%writefile evaluate_hgp_metrics.py
"""Test metrics of the trained DSTFGCN in HGP's format (hgp/metrics.py), from the authors' best checkpoint.

    python evaluate_hgp_metrics.py --config_path config/PEMS-BAY_hgp.json --device cuda:0 --out results

The authors' train.py reports MAE / MAPE / RMSE averaged over all 12 steps, with MAPE over every non-zero target. HGP reports steps 3 / 6 / 12
(15 / 30 / 60 min), MAPE over targets >= 1, and averages the three horizons. Here a test window counts at a horizon when every sensor is
observed (> 0) at that step - HGP's rule - and the metrics are computed exactly as in hgp/metrics.py.
"""
import argparse
import glob
import json
import os
import pickle
import sys

import numpy as np
import torch

sys.path.insert(0, ".")
import util
from engine import trainer

ap = argparse.ArgumentParser()
ap.add_argument("--config_path", required=True)
ap.add_argument("--device", default="cuda:0")
ap.add_argument("--out", default="results")
ap.add_argument("--mape_min", type=float, default=1.0)
a = ap.parse_args()

config = json.load(open(a.config_path))
data = util.load_dataset(config["data"], config["N_t"], config["batch_size"], config["batch_size"], config["batch_size"])
adj = pickle.load(open(config["adjacency matrix"], "rb"), encoding="latin1")[2]
device = torch.device(a.device)
model = trainer(config, data["scaler"], adj, a.device).model
ckpt = sorted(glob.glob(config["save"] + f"DSTFGCN_exp{config['expid']}_best_*.pth"), key=os.path.getmtime)[-1]
model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()
print("checkpoint", ckpt)

realy = torch.Tensor(data["y_test"]).to(device).transpose(1, 3)[:, 0, :, :]          # [W, N, 12]
outputs = []
for x, y, d in data["test_loader"].get_iterator():
    with torch.no_grad():
        outputs.append(model(torch.Tensor(x).to(device).transpose(1, 3), d).transpose(1, 3).squeeze(1))
yhat = torch.cat(outputs, 0)[: realy.size(0)]
pred = data["scaler"].inverse_transform(yhat).cpu().numpy().astype(np.float64)        # [W, N, 12]
real = realy.cpu().numpy().astype(np.float64)


def evaluate(t, p):
    err = p - t
    keep = np.abs(t) >= a.mape_min
    return {"mae": float(np.mean(np.abs(err))), "rmse": float(np.sqrt(np.mean(err ** 2))),
            "mape": float(100 * np.mean(np.abs(err[keep]) / np.abs(t[keep]))) if keep.any() else float("nan"), "n": int(len(t))}


rows, saved = {}, {}
for h in (3, 6, 12):
    t, p = real[:, :, h - 1], pred[:, :, h - 1]
    ok = (t > 0).all(axis=1)
    rows[h] = evaluate(t[ok], p[ok])
    saved[f"targets_h{h}"], saved[f"predictions_h{h}"] = t[ok].astype(np.float32), p[ok].astype(np.float32)
    print(f"h={h:2d} ({5 * h:2d} min)  MAE {rows[h]['mae']:.3f}  RMSE {rows[h]['rmse']:.3f}  MAPE {rows[h]['mape']:.2f}%  windows {rows[h]['n']}")
rows["mean"] = {k: float(np.mean([rows[h][k] for h in (3, 6, 12)])) for k in ("mae", "rmse", "mape")}
rows["error_growth_rmse"] = rows[12]["rmse"] / rows[3]["rmse"]
print("mean of h3/h6/h12:", rows["mean"], "| RMSE growth h12/h3:", round(rows["error_growth_rmse"], 3))
valid = real > 0
e = (pred - real)[valid]
rows["all_12_steps"] = {"mae": float(np.abs(e).mean()), "rmse": float(np.sqrt((e ** 2).mean()))}
print("pooled over all 12 steps (entries > 0):", rows["all_12_steps"])
os.makedirs(a.out, exist_ok=True)
json.dump(rows, open(os.path.join(a.out, "test_metrics.json"), "w"), indent=2)
np.savez_compressed(os.path.join(a.out, "test_predictions.npz"), **saved)


## 2. Config: HGP common settings, everything else the authors' PEMS-BAY values

In [ ]:
cfg = {
    "dataset_class": "PEMS-BAY",
    "data": "./data/PEMS-BAY",
    "adjacency matrix": "./data/graph/adj_PEMS-BAY.pkl",
    "input_dim": 1,
    "output_dim": 12,
    "train_rate": 0.7,
    "eval_rate": 0.1,
    "test_rate": 0.2,
    "batch_size": 16,
    "layers": 3,
    "N_t": 2016,
    "temporal_dim": [
        10,
        10,
        4,
        4
    ],
    "max_epoch": 40,
    "patience": 6,
    "hidden_dimension": 64,
    "embedding_dimension": 10,
    "num_nodes": 325,
    "learning_rate": 0.0003,
    "learning_rate_decay": 0.5,
    "lr_decay_milestones": [
        40,
        80,
        100
    ],
    "drop": 0.1,
    "weight_decay": 0.0001,
    "print_every": 600,
    "seed": 42,
    "save": "./save/",
    "expid": 1,
    "set_loss": "huber"
}
cfg['max_epoch'] = MAX_EPOCH
cfg['seed'] = SEED
os.makedirs('config', exist_ok=True)
json.dump(cfg, open('config/PEMS-BAY_hgp.json', 'w'), indent=4)
print(json.dumps(cfg))

## 3. Data

In [ ]:
def run(cmd):
    with subprocess.Popen(cmd, cwd=CODE, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as pr:
        for line in pr.stdout:
            print(line, end="")
    assert pr.returncode == 0, f"{cmd[1]} exited with {pr.returncode}"

run([sys.executable, "prepare_pemsbay.py", "--traffic", TRAFFIC, "--adjacency", ADJ])

## 4. Train
One epoch is 2,279 batches; the log prints the seconds per epoch - check the first epoch against the 12-hour session limit (40 epochs maximum, early stopping with patience 6). The authors' code saves a checkpoint every epoch and keeps the one with the lowest validation loss.

In [ ]:
run([sys.executable, "run_hgp_protocol.py", "--config_path", "config/PEMS-BAY_hgp.json", "--device", "cuda:0"])

## 5. Test metrics in HGP's format (steps 3 / 6 / 12 = 15 / 30 / 60 min)

In [ ]:
out = f"{WORK}/dstfgcn_PEMSBAY"
run([sys.executable, "evaluate_hgp_metrics.py", "--config_path", "config/PEMS-BAY_hgp.json", "--device", "cuda:0", "--out", out])
for f in glob.glob("log/*.log") + ["config/PEMS-BAY_hgp.json"]:
    shutil.copy(f, out)
best = sorted(glob.glob("save/DSTFGCN_exp1_best_*.pth"), key=os.path.getmtime)[-1]
shutil.copy(best, out)